In [ ]:
#############################
# 반드시 먼저 실행해주세요! #
#############################
import subprocess

def run_git_safe(args: list, timeout: int = 30) -> tuple[bool, str]:
    '''
    Git 명령어를 안전하게 실행합니다. (intermediate-project 디렉토리 고정)

    Args:
        args (list): Git 명령어 인자 리스트. 예: ["status"], ["add", "main.py"]
        timeout (int): 타임아웃 초. 기본값 30. push/pull/clone 은 120 권장.

    Returns:
        tuple[bool, str]: (성공 여부, stdout 내용)
    '''
    try:
        result = subprocess.run(
            ["git", "-C", "intermediate-project"] + args, capture_output=True, text=True, check=True, timeout=timeout)
        # 성공이더라도 경고(warning)는 stderr 로 전달되므로 출력
        if result.stderr:
            print(f"[경고]\n{result.stderr.strip()}")
        if result.stdout:
            print(result.stdout)
        return True, result.stdout.strip()
    except subprocess.TimeoutExpired:
        print("[장애 조치] 명령어 실행 시간 초과. (인증 대기 또는 네트워크 문제)")
        return False, ""
    except subprocess.CalledProcessError as e:
        print(f"\n[실행 실패] git {' '.join(args)}")
        print(f"[상세 에러]\n{e.stderr.strip()}")
        if "conflict" in e.stderr.lower():
            print("\n[조치 가이드] 파일 충돌(Conflict)이 발생했습니다! 팀장에게 문의하세요.")
        elif "commit your changes or stash them" in e.stderr.lower():
            print("\n[조치 가이드] 저장하지 않은 작업이 있습니다. 먼저 커밋 셀을 실행하세요.")
        return False, ""
    except FileNotFoundError:
        print("[장애 조치] Git이 설치되어 있지 않거나 경로를 찾을 수 없습니다.")
        return False, ""
    except Exception as e:
        print(f"[알 수 없는 오류] {e}")
        return False, ""

print("run_git_safe 함수가 정의되었습니다.")

In [ ]:
###########################################
# ↓↓↓ 내 작업을 시작할 때 사용합니다! ↓↓↓ #
###########################################
# 반드시 맨 위의 'run_git_safe 정의' 셀을 먼저 실행하세요!
# 1, 2 순서대로 실행해주세요

In [ ]:
# 1. 모든 branch의 내용을 최신화합니다.

run_git_safe(["fetch", "--all", "--prune"], timeout=60)

In [ ]:
# 2. 내 작업이 시작될 브랜치를 생성합니다.

my_branch = ""      # ← 생성할 브랜치 이름을 입력하세요
target_branch = ""  # ← 시작 기준 브랜치를 입력하세요 (예: origin/main)

if not my_branch.strip() or not target_branch.strip():
    print("my_branch와 target_branch를 모두 입력해주세요.")
else:
    run_git_safe(["checkout", "-b", my_branch, target_branch])

In [ ]:
############################################
# ↓↓↓ 내 작업을 동기화할 때 사용합니다! ↓↓↓ #
############################################
# 반드시 맨 위의 'run_git_safe 정의' 셀을 먼저 실행하세요!
# 1 ~ 5 순서대로 실행해주세요

In [ ]:
# 1. 현재 브랜치와 변경사항을 확인합니다.

ok, stdout = run_git_safe(["branch", "--show-current"])
current_branch = stdout if ok else ""

print(f"현재 브랜치: {current_branch}")
run_git_safe(["status"])

In [ ]:
# 2. 수정된 파일과 변경 규모를 확인합니다.

run_git_safe(["diff", "--stat"])

In [ ]:
# 3. files에 기록한 파일을 모두 커밋 대상으로 등록합니다.

# 여러 파일 등록 방법: ["myfile0.txt", "myfile1.txt", ...]
files = [""]  # ← 파일 경로를 입력하세요

file_args = [f for f in files if f.strip()]
if not file_args:
    print("파일 경로를 입력해주세요.")
else:
    run_git_safe(["add"] + file_args)

In [ ]:
# 4. 현재 브랜치에 커밋 대상을 기록합니다.

commit_message = ""  # ← 커밋 메시지를 입력하세요

if not commit_message.strip():
    print("커밋 메시지가 비어있습니다.")
else:
    run_git_safe(["commit", "-m", commit_message])

In [ ]:
# 5. 내 변경사항을 GitHub에 업로드합니다.

if not current_branch:
    print("현재 브랜치를 확인할 수 없습니다. 1번 셀을 먼저 실행하세요.")
elif current_branch == "main":
    print("main 브랜치에서는 직접 push할 수 없습니다.")
else:
    # --set-upstream: 첫 push 시 원격 브랜치를 자동으로 연결합니다
    run_git_safe(["push", "--set-upstream", "origin", current_branch], timeout=120)

In [ ]:
######################################
# ↓↓↓ 처음 실행할 때 사용합니다! ↓↓↓ #
######################################
# 1, 2 순서대로 실행해주세요

In [ ]:
# 1. 최초 실행이나, 프로젝트 폴더가 없을 경우 1회 실행해서 git 저장소와 연동합니다.

import os
import subprocess  # run_git_safe 정의 셀을 실행하지 않은 경우를 대비

if os.path.exists("intermediate-project"):
    print("이미 프로젝트 폴더가 존재합니다. clone을 건너뜁니다.")
else:
    try:
        # clone은 intermediate-project 폴더 외부에서 실행하므로 run_git_safe 미사용
        result = subprocess.run(
            ["git", "clone", "https://github.com/Codeit-AI10-Part3-4Team/intermediate-project"],
            capture_output=True,
            text=True,
            check=True,
            timeout=120,
        )
        print("clone 완료.")
        if result.stdout:
            print(result.stdout)
    except subprocess.TimeoutExpired:
        print("[장애 조치] 네트워크 연결을 확인하세요. (120초 초과)")
    except subprocess.CalledProcessError as e:
        print(f"[실행 실패]\n{e.stderr.strip()}")

In [ ]:
# 2. 최초 실행 후, 작업자 등록을 진행합니다. 1회만 실행.

import subprocess  # run_git_safe 정의 셀을 실행하지 않은 경우를 대비

user_email = ""  # ← GitHub에 등록한 이메일
user_name = ""   # ← GitHub에 등록한 유저 이름

if not user_email.strip() or not user_name.strip():
    print("이메일이나 유저 이름이 비어 있습니다.")
else:
    try:
        # git config --global 은 프로젝트 경로와 무관하므로 run_git_safe 미사용
        subprocess.run(["git", "config", "--global", "user.email", user_email], check=True)
        subprocess.run(["git", "config", "--global", "user.name", user_name], check=True)
        print("작업자 등록 완료.")
    except subprocess.CalledProcessError as e:
        print(f"[실행 실패] 작업자 등록 중 오류가 발생했습니다.\n{e.stderr}")

In [ ]:
#############################################################################
# 상황에 맞춰 사용하세요. 일부 위험한 기능도 있으니 신중히 사용을 결정하세요! #
#############################################################################
# 반드시 맨 위의 'run_git_safe 정의' 셀을 먼저 실행하세요!
# 되도록 1, 2번을 실행해서 현재 상황을 확인한 후에 사용하시기 바랍니다

In [ ]:
# 1. 최근 커밋 기록을 확인합니다.

run_git_safe(["log", "--oneline", "-10"])

In [ ]:
# 2. 수정된 파일 목록을 확인합니다.

run_git_safe(["diff", "--name-only"])

In [ ]:
# 3. 원격의 최신 변경사항을 내 브랜치에 반영합니다.
# (다른 환경에서 push한 내용을 받아올 때 사용하세요)

if not current_branch:
    print("현재 브랜치를 확인할 수 없습니다. 동기화 섹션의 1번 셀을 먼저 실행하세요.")
elif current_branch == "main":
    print("main 브랜치에서는 pull하지 마세요. 작업 브랜치로 이동하세요.")
else:
    run_git_safe(["pull", "origin", current_branch], timeout=120)

In [ ]:
# 4. 현재 작업 중인 변경사항을 임시 저장합니다.
# (커밋하지 않고 브랜치를 이동해야 할 때 사용하세요)

run_git_safe(["stash", "push", "-m", "임시저장"])
print("브랜치 이동 후 작업을 복원하려면 아래 셀(5번)을 실행하세요.")

In [ ]:
# 5. 임시 저장한 변경사항을 복원합니다.

run_git_safe(["stash", "pop"])

In [ ]:
# 경고 ####################################
# 커밋하지 않은 수정 내용을 모두 삭제합니다.

# 여러 파일 등록 방법: ["myfile0.txt", "myfile1.txt", ...]
files = [""]  # ← 파일 경로를 입력하세요

file_args = [f for f in files if f.strip()]
if not file_args:
    print("파일 경로를 입력해주세요.")
else:
    run_git_safe(["restore"] + file_args)